In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
!pip install --upgrade pymilvus sentence-transformers huggingface-hub langchain_community langchain-text-splitters pymupdf tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 71.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 23.4 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully uninstalled tqdm-4.67.1
  Attempting uninstall: huggingface-hub
 

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_API_KEY")

In [4]:
import os

os.environ["HF_TOKEN"] = hf_token

In [5]:
!pip install pymilvus[milvus_lite]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 33.4 MB/s eta 0:00:00:00:0100:01


In [6]:
!pip install langchain langchain-community pypdf pymupdf faiss-cpu sentence-transformers huggingface_hub

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0rc2
    Uninstalling packaging-26.0rc2:
      Successfully uninstalled packaging-26.0rc2
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.1
    Uninstalling langchain-text-splitt

In [8]:
# =============================
# IMPORTS
# =============================
import gradio as gr
import json
from tqdm import tqdm

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient
from huggingface_hub import InferenceClient


# =============================
# 1️⃣ LOAD PDF DATASET
# =============================
PDF_PATH = "/kaggle/input/os-by-silberschatz/Meta-Reports-Fourth-Quarter-and-Full-Year-2025-Results-2026.pdf"

loader = PyMuPDFLoader(PDF_PATH, extract_images=False)
docs = loader.load()

print(f"Loaded {len(docs)} pages")


# =============================
# 2️⃣ SPLIT INTO CHUNKS
# =============================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(docs)
text_lines = [chunk.page_content for chunk in chunks]

print(f"Created {len(text_lines)} chunks")


# =============================
# 3️⃣ LOAD EMBEDDING MODEL
# =============================
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

def emb_text(text):
    return embedding_model.encode(
        [text],
        normalize_embeddings=True
    ).tolist()[0]

# Get embedding dimension
embedding_dim = len(emb_text("test"))
print("Embedding dimension:", embedding_dim)


# =============================
# 4️⃣ SETUP MILVUS VECTOR DB
# =============================
milvus_client = MilvusClient(uri="./hf_milvus_demo.db")

collection_name = "rag_collection"

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",
    consistency_level="Strong",
)


# =============================
# 5️⃣ CREATE & INSERT EMBEDDINGS
# =============================
data = []

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({
        "id": i,
        "vector": emb_text(line),
        "text": line
    })

milvus_client.insert(collection_name=collection_name, data=data)

print("Embeddings inserted into Milvus")


# =============================
# 6️⃣ LOAD LLM (LLAMA 3.1)
# =============================
repo_id = "meta-llama/Llama-3.1-8B-Instruct"
llm_client = InferenceClient(model=repo_id, timeout=120)


# =============================
# 7️⃣ PROMPT TEMPLATE
# =============================
PROMPT = """
Use the following pieces of information enclosed in <context> tags 
to answer the question enclosed in <question> tags.

If the answer is not in the context, say you don't know.

<context>
{context}
</context>

<question>
{question}
</question>
"""

Loaded 10 pages
Created 29 chunks
Embedding dimension: 384


Creating embeddings: 100%|██████████| 29/29 [00:00<00:00, 83.57it/s]

Embeddings inserted into Milvus


In [9]:
# =============================
# 8️⃣ RAG FUNCTION FOR GRADIO
# =============================
def ask_question(question, history):

    # Embed user question
    question_embedding = emb_text(question)

    # Search in Milvus
    search_res = milvus_client.search(
        collection_name=collection_name,
        data=[question_embedding],
        limit=3,
        search_params={"metric_type": "IP", "params": {}},
        output_fields=["text"],
    )

    # Extract retrieved chunks
    retrieved_chunks = [
        res["entity"]["text"] for res in search_res[0]
    ]

    context = "\n\n".join(retrieved_chunks)

    # Format prompt
    prompt = PROMPT.format(context=context, question=question)

    # Call LLM
    response = llm_client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1000,
    )

    answer = response.choices[0].message.content.strip()

    return answer


# =============================
# 9️⃣ GRADIO UI
# =============================
demo = gr.ChatInterface(
    fn=ask_question,
    title="📘 RAG Assistant",
    description="Ask questions about the uploaded PDF dataset.",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://1801fabadd837380f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
